# 🤖 Dia 1 — Conectando a IA ao Mundo Real

Bem-vindo ao primeiro dia do curso! Hoje vamos:

1. **Verificar a conexão** com o proxy LLM e o servidor de serviços
2. **Explorar a API** de e-mail e agenda manualmente
3. **Criar ferramentas (tools)** em LangChain para cada operação
4. **Montar um agente multi-tool** que entende linguagem natural e decide sozinho qual ferramenta usar

---

> 💡 **Conceito central do dia:** *Function Calling* — a capacidade do LLM de reconhecer quando precisa agir no mundo real e chamar a função certa com os parâmetros certos.

---
## Parte 1 — Configuração e Testes de Conexão

In [ ]:
# Célula 1 — Instalar dependências
!pip install -q langchain-anthropic langchain langgraph

In [ ]:
# Célula 2 — Configuração central
# ⚠️  Preencha com seus dados antes de continuar

PROXY_URL   = "https://interview-server-mocado.b60gda.easypanel.host/"  # proxy LLM do curso
ALUNO_TOKEN = "xpto_aluno-01"   # token do proxy fornecido pelo professor

API_URL     = "https://interview-email-server.b60gda.easypanel.host/"   # servidor de email + agenda
EMAIL_TOKEN = "aluno-01"        # token de autenticação na API (aluno-01 … aluno-10)
EMAIL_SENHA = "1234"

In [ ]:
# Célula 3 — Health check do servidor de serviços
import requests

r = requests.get(f"{API_URL}/health")
print(r.status_code, r.json())

In [ ]:
# Célula 4 — Teste de conexão com o LLM via LangChain
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=128,
)

resp = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print(resp.content)

---
## Parte 2 — Explorando a API manualmente

Antes de criar o agente, vamos entender o que a API oferece fazendo chamadas diretas. Isso é importante para saber exatamente o que as nossas *ferramentas* precisarão fazer.

In [ ]:
# Célula 5 — Login e criação dos headers de autenticação
r = requests.post(f"{API_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_SENHA,
})
print(r.status_code, r.json())

AUTH = {"Authorization": f"Bearer {EMAIL_TOKEN}"}

In [ ]:
# Célula 6 — Listar usuários disponíveis (útil para saber para quem enviar e-mails)
r = requests.get(f"{API_URL}/users", headers=AUTH)
for u in r.json():
    print(f"  {u['name']:10}  →  {u['email']}")

In [ ]:
# Célula 7 — Caixa de entrada
r = requests.get(f"{API_URL}/emails/inbox", headers=AUTH)
inbox = r.json()
print(f"{inbox['count']} mensagem(ns) para {inbox['email']}")
for msg in inbox["messages"]:
    print(f"  [{msg['id'][:8]}…]  De: {msg['from']}  |  {msg['subject']}")

In [ ]:
# Célula 8 — Enviar um e-mail manualmente (teste)
r = requests.post(f"{API_URL}/emails/send", headers=AUTH, json={
    "to": "aluno02@curso.ia",
    "subject": "Teste manual",
    "body": "Olá! Enviando este e-mail de teste direto pela API.",
})
print(r.status_code, r.json())

In [ ]:
# Célula 9 — Listar compromissos da agenda
r = requests.get(f"{API_URL}/agenda", headers=AUTH)
agenda = r.json()
print(f"{agenda['count']} compromisso(s) para {agenda['email']}")
for c in agenda["compromissos"]:
    print(f"  [{c['id'][:8]}…]  {c['data']}  {c['hora_inicio']}–{c['hora_fim']}  |  {c['titulo']}")

In [ ]:
# Célula 10 — Criar um compromisso manualmente (teste)
r = requests.post(f"{API_URL}/agenda", headers=AUTH, json={
    "titulo": "Reunião de teste",
    "descricao": "Criado diretamente pela API para testar o endpoint.",
    "data": "2025-05-05",
    "hora_inicio": "14:00",
    "hora_fim": "15:00",
})
print(r.status_code, r.json())

In [ ]:
# Célula 11 — Testar erro de conflito de horário (tente criar outro no mesmo horário)
r = requests.post(f"{API_URL}/agenda", headers=AUTH, json={
    "titulo": "Vai conflitar!",
    "data": "2025-05-05",
    "hora_inicio": "14:30",  # Conflita com 14:00–15:00
    "hora_fim": "15:30",
})
print(r.status_code)
print(r.json())  # Deve retornar 409 com detalhes do conflito

---
## Parte 3 — Criando as Ferramentas (Tools)

Agora transformamos cada operação da API em uma **ferramenta LangChain**.

> 💡 **Por que isso importa?** O LLM não consegue chamar APIs diretamente. Nós criamos funções Python decoradas com `@tool` e o LangChain as descreve ao modelo — que então decide *quando* e *como* chamá-las com base no que o usuário pediu.

In [ ]:
# Célula 12 — Login para o agente (headers reutilizados por todas as ferramentas)
import requests
from langchain_core.tools import tool
from typing import Optional

_login = requests.post(f"{API_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_SENHA,
})
assert _login.status_code == 200, f"Login falhou: {_login.json()}"
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Login OK →", _login.json())

In [ ]:
# Célula 13 — Ferramentas de E-mail

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Use esta ferramenta quando o usuário pedir para enviar ou encaminhar um e-mail.

    Args:
        to: E-mail do destinatário. Formato: alunoXX@curso.ia (ex: aluno02@curso.ia).
            Aceita múltiplos separados por vírgula.
        subject: Assunto do e-mail.
        body: Corpo completo do e-mail.
        cc: E-mail(s) em cópia, separados por vírgula — opcional.
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc
    r = requests.post(f"{API_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"✅ E-mail enviado com sucesso para '{to}'."
    return f"❌ Erro ao enviar e-mail ({r.status_code}): {r.json()}"


@tool
def list_inbox(data: Optional[str] = None) -> str:
    """Lista as mensagens recebidas na caixa de entrada.

    Use quando o usuário quiser ver seus e-mails, verificar mensagens recebidas
    ou checar se recebeu algo.

    Args:
        data: Data para filtrar no formato YYYY-MM-DD — opcional.
              Se omitido, retorna todas as mensagens.
    """
    params = {}
    if data:
        params["data"] = data
    r = requests.get(f"{API_URL}/emails/inbox", headers=_headers, params=params)
    inbox = r.json()
    if inbox["count"] == 0:
        return "📭 Caixa de entrada vazia."
    linhas = [f"📬 {inbox['count']} mensagem(ns):"]
    for msg in inbox["messages"]:
        lida = "✓" if msg.get("read") else "●"
        linhas.append(f"  {lida} [{msg['id'][:8]}…] De: {msg['from']} | {msg['subject']}")
    return "\n".join(linhas)


@tool
def list_sent() -> str:
    """Lista os e-mails enviados pelo usuário.

    Use quando o usuário quiser ver os e-mails que já enviou ou
    verificar se uma mensagem foi enviada anteriormente.
    """
    r = requests.get(f"{API_URL}/emails/sent", headers=_headers)
    sent = r.json()
    if sent["count"] == 0:
        return "📤 Nenhum e-mail enviado ainda."
    linhas = [f"📤 {sent['count']} e-mail(s) enviado(s):"]
    for msg in sent["messages"]:
        destinatarios = ", ".join(msg["to"]) if isinstance(msg["to"], list) else msg["to"]
        linhas.append(f"  [{msg['id'][:8]}…] Para: {destinatarios} | {msg['subject']} | {msg['timestamp']}")
    return "\n".join(linhas)


print("Ferramentas de e-mail criadas: send_email, list_inbox, list_sent")

In [ ]:
# Célula 14 — Ferramentas de Agenda

@tool
def create_appointment(
    titulo: str,
    data: str,
    hora_inicio: str,
    hora_fim: str,
    descricao: Optional[str] = None,
) -> str:
    """Cria um novo compromisso na agenda do usuário.

    Use esta ferramenta quando o usuário pedir para agendar, marcar ou criar
    um compromisso, reunião, evento ou lembrete com data e hora definidos.

    IMPORTANTE: Se o usuário não informar a hora de término, estime 1 hora após
    o início. Se não informar a data, pergunte antes de chamar esta ferramenta.

    Args:
        titulo: Título ou nome do compromisso (obrigatório).
        data: Data no formato YYYY-MM-DD (obrigatório).
        hora_inicio: Hora de início no formato HH:MM (obrigatório).
        hora_fim: Hora de término no formato HH:MM (obrigatório, deve ser após hora_inicio).
        descricao: Descrição ou observações adicionais — opcional.
    """
    payload = {
        "titulo": titulo,
        "data": data,
        "hora_inicio": hora_inicio,
        "hora_fim": hora_fim,
    }
    if descricao:
        payload["descricao"] = descricao

    r = requests.post(f"{API_URL}/agenda", headers=_headers, json=payload)

    if r.status_code == 201:
        c = r.json()
        return (
            f"✅ Compromisso criado com sucesso!\n"
            f"   Título: {c['titulo']}\n"
            f"   Data:   {c['data']} das {c['hora_inicio']} às {c['hora_fim']}\n"
            f"   ID:     {c['id']}"
        )
    if r.status_code == 409:
        erro = r.json()
        return (
            f"⚠️  Conflito de horário! {erro.get('mensagem', '')}\n"
            f"   Tente um horário diferente."
        )
    return f"❌ Erro ao criar compromisso ({r.status_code}): {r.json()}"


@tool
def list_agenda(data: Optional[str] = None, data_ini: Optional[str] = None, data_fim: Optional[str] = None) -> str:
    """Lista os compromissos da agenda do usuário.

    Use quando o usuário quiser ver sua agenda, verificar compromissos,
    checar disponibilidade ou consultar o que tem marcado.

    Args:
        data: Data exata para filtrar, formato YYYY-MM-DD — opcional.
        data_ini: Início de um intervalo de datas, formato YYYY-MM-DD — opcional.
        data_fim: Fim de um intervalo de datas, formato YYYY-MM-DD — opcional.
              Se omitidos, retorna todos os compromissos.
    """
    params = {}
    if data:
        params["data"] = data
    if data_ini:
        params["data_ini"] = data_ini
    if data_fim:
        params["data_fim"] = data_fim

    r = requests.get(f"{API_URL}/agenda", headers=_headers, params=params)
    agenda = r.json()

    if agenda["count"] == 0:
        filtro = f" para {data or f'{data_ini} a {data_fim}'}" if (data or data_ini) else ""
        return f"📅 Nenhum compromisso encontrado{filtro}."

    linhas = [f"📅 {agenda['count']} compromisso(s):"]
    for c in agenda["compromissos"]:
        linhas.append(
            f"  • [{c['id'][:8]}…] {c['data']}  {c['hora_inicio']}–{c['hora_fim']}  |  {c['titulo']}"
            + (f"\n      {c['descricao']}" if c.get('descricao') else "")
        )
    return "\n".join(linhas)


print("Ferramentas de agenda criadas: create_appointment, list_agenda")

---
## Parte 4 — O Agente Multi-Tool

Agora montamos o agente que tem acesso a **todas as ferramentas** e decide por conta própria qual usar.

> 💡 **Observe o System Prompt:** ele define a *personalidade* e as *regras de negócio* do agente. A qualidade desse prompt impacta diretamente o comportamento — este é um dos temas centrais do curso!

In [ ]:
# Célula 15 — Criar o agente com todas as ferramentas
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from datetime import date

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=1024,
)

HOJE = date.today().isoformat()

system_prompt = f"""\
Você é um assistente de produtividade pessoal. Hoje é {HOJE}.

Você tem acesso às seguintes ferramentas:
- send_email: envia e-mails
- list_inbox: consulta a caixa de entrada
- list_sent: consulta os e-mails enviados
- create_appointment: cria compromissos na agenda
- list_agenda: consulta compromissos da agenda

Regras importantes:
- E-mails válidos seguem o padrão alunoXX@curso.ia (ex: aluno02@curso.ia, aluno10@curso.ia)
- Datas devem estar no formato YYYY-MM-DD
- Horários devem estar no formato HH:MM
- Se o usuário não informar um dado obrigatório, pergunte antes de chamar a ferramenta
- Se uma ação falhar, explique o erro ao usuário em linguagem simples e sugira como corrigir
- Sempre confirme as ações realizadas de forma clara e resumida
"""

ferramentas = [send_email, list_inbox, list_sent, create_appointment, list_agenda]

agente = create_agent(
    model=llm,
    tools=ferramentas,
    system_prompt=system_prompt,
)

print(f"✅ Agente criado com {len(ferramentas)} ferramentas: {[t.name for t in ferramentas]}")

In [ ]:
# Célula 16 — Função auxiliar para invocar o agente de forma limpa
def perguntar(prompt: str, verbose: bool = False) -> str:
    """Envia uma mensagem ao agente e retorna a resposta final."""
    result = agente.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 15},
    )
    msgs = result["messages"]
    if verbose:
        for m in msgs:
            print(f"[{m.__class__.__name__}] {m.content}\n")
    return msgs[-1].content

print("Função 'perguntar()' pronta. Use: perguntar('seu pedido aqui')")

---
## Parte 5 — Experimentando o Agente

Teste os exemplos abaixo e depois crie os seus próprios!

In [ ]:
# Exemplo A — Enviar e-mail simples
resposta = perguntar(
    "Manda um e-mail para aluno03@curso.ia avisando que a reunião de amanhã "
    "foi cancelada e que vamos reagendar em breve."
)
print(resposta)

In [ ]:
# Exemplo B — Agendar um compromisso
resposta = perguntar(
    "Agenda uma reunião de planejamento para 2025-05-07 das 10h às 11h30. "
    "Coloca a descrição: 'Revisão do roadmap do trimestre'."
)
print(resposta)

In [ ]:
# Exemplo C — Consultar a agenda
resposta = perguntar("O que tenho na minha agenda esta semana?")
print(resposta)

In [ ]:
# Exemplo E — Tentativa de conflito (o agente deve tratar o erro)
resposta = perguntar(
    "Marca uma reunião rápida para 2025-05-07 às 10h30 com duração de 1 hora."
    # Deve conflitar com o Exemplo B se executado na mesma sessão
)
print(resposta)

In [ ]:
# ✏️  Espaço livre — teste seus próprios prompts!
meu_pedido = """  ← escreva aqui o que quiser """

resposta = perguntar(meu_pedido)
print(resposta)

---
## 🔍 Modo Verbose — veja o raciocínio do agente

Passe `verbose=True` para ver cada passo do agente: o que ele pensou, qual ferramenta chamou e o que recebeu de volta.

In [ ]:
# Modo verbose: observe o ciclo Pensamento → Ação → Observação → Resposta
resposta = perguntar(
    "Agenda um café para 2025-05-09 das 09h às 09h30 e avisa o aluno02@curso.ia.",
    verbose=True,
)

---
## ✅ Resumo do Dia 1

| O que fizemos | Por que importa |
|---|---|
| Conectamos ao proxy LLM | O modelo fica no servidor do curso — sem custo de API própria |
| Exploramos a API manualmente | Entender a API é o primeiro passo para criar boas ferramentas |
| Criamos 4 tools com `@tool` | O decorator transforma uma função Python em algo que o LLM pode chamar |
| Montamos um agente multi-tool | O agente decide sozinho qual ferramenta usar baseado no pedido |
| Testamos ações compostas | O agente encadeia múltiplas ferramentas para resolver tarefas complexas |

### 🚀 O que vem a seguir (Dia 2)
No próximo dia vamos explorar **como o agente raciocina**, trabalhar a fundo o design de *system prompts* e adicionar lógica de verificação de conflitos **antes** de agendar.
